In [5]:
# @title Cell 1: Mount Drive & Imports
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# @title Cell 2: Load Processed Dataset
DATA_PATH = '/content/drive/MyDrive/Tech Fem team - SIC capstone project/Week 1/Data Pipeline/data/processed/crude_weekly_complete.parquet'

df = pd.read_parquet(DATA_PATH)

if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

TARGET_COL = 'target_next_week_change'
IGNORE_COLS = ['date', TARGET_COL, 'event_name', 'stocks_change_outlier']
FEATURE_COLS = [col for col in df.columns if col not in IGNORE_COLS]

df_clean = df.dropna(subset=[TARGET_COL] + FEATURE_COLS).reset_index(drop=True)

TEST_SIZE = 104
train_df = df_clean.iloc[:-TEST_SIZE].copy()

X_train = train_df[FEATURE_COLS]
y_train_raw = train_df[TARGET_COL]

# Calculate target statistics for scaling
y_mean = y_train_raw.mean()
y_std = y_train_raw.std()
y_train_scaled = (y_train_raw - y_mean) / y_std

print(f"Train Dataset Loaded for Walk-Forward Validation: {X_train.shape[0]} rows")

Train Dataset Loaded for Walk-Forward Validation: 1067 rows


In [7]:
# @title Cell 3: Execute 5-Fold Walk-Forward Cross-Validation
tscv = TimeSeriesSplit(n_splits=5)

# Half-life exponential decay setup
half_life_weeks = 156
decay_rate = np.log(2) / half_life_weeks

wf_results = []

print("=== Starting 5-Fold Walk-Forward Validation (LightGBM) ===")

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train_scaled.iloc[train_idx]
    X_va, y_va = X_train.iloc[val_idx], y_train_scaled.iloc[val_idx]

    # Recency weights recalculated for each expanding training fold
    t_idx = np.arange(len(X_tr))
    w_tr = np.exp(-decay_rate * (t_idx[-1] - t_idx))

    model = lgb.LGBMRegressor(
        learning_rate=0.01,
        max_depth=3,
        num_leaves=7,
        min_child_samples=20,
        n_estimators=200,
        random_state=42,
        verbose=-1
    )
    model.fit(X_tr, y_tr, sample_weight=w_tr)

    # Predict on unseen future validation fold
    preds_scaled = model.predict(X_va)


    # Unscale predictions back to barrel units (kb)
    preds = (preds_scaled * y_std) + y_mean
    y_actual = y_train_raw.iloc[val_idx].values  # Use true unscaled ground truth directly




    mae = mean_absolute_error(y_actual, preds)
    rmse = np.sqrt(mean_squared_error(y_actual, preds))
    r2 = r2_score(y_actual, preds)
    dir_acc = np.mean(np.sign(preds) == np.sign(y_actual)) * 100

    wf_results.append({
        'Fold': fold + 1,
        'Train_Size': len(X_tr),
        'Val_Size': len(X_va),
        'MAE_kb': mae,
        'RMSE_kb': rmse,
        'R2_Score': r2,
        'Directional_Acc_%': dir_acc
    })

    print(f"Fold {fold+1} | Train: {len(X_tr)} wks | Val: {len(X_va)} wks | MAE: {mae:.2f} kb | RMSE: {rmse:.2f} kb | R²: {r2:.4f} | Dir. Acc: {dir_acc:.1f}%")

wf_df = pd.DataFrame(wf_results)

=== Starting 5-Fold Walk-Forward Validation (LightGBM) ===
Fold 1 | Train: 182 wks | Val: 177 wks | MAE: 2826.48 kb | RMSE: 3575.57 kb | R²: 0.0752 | Dir. Acc: 58.8%
Fold 2 | Train: 359 wks | Val: 177 wks | MAE: 2984.24 kb | RMSE: 3747.25 kb | R²: 0.1389 | Dir. Acc: 63.8%
Fold 3 | Train: 536 wks | Val: 177 wks | MAE: 3621.27 kb | RMSE: 4624.19 kb | R²: 0.0663 | Dir. Acc: 67.8%
Fold 4 | Train: 713 wks | Val: 177 wks | MAE: 4637.03 kb | RMSE: 5834.54 kb | R²: 0.0720 | Dir. Acc: 58.8%
Fold 5 | Train: 890 wks | Val: 177 wks | MAE: 4026.01 kb | RMSE: 5130.58 kb | R²: 0.0896 | Dir. Acc: 63.3%


In [8]:
# @title Cell 4: Save Walk-Forward Results to Drive
OUTPUT_DIR = '/content/drive/MyDrive/Tech Fem team - SIC capstone project/Week 2/4. Model Training and Optimization/4.5 Walk forward validation'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save Summary Table
wf_df.to_csv(os.path.join(OUTPUT_DIR, 'walk_forward_lightgbm_summary.csv'), index=False)

# Print Mean Aggregates
print("\n=== Walk-Forward Validation Overall Averages ===")
print(f"Mean MAE : {wf_df['MAE_kb'].mean():.2f} kb")
print(f"Mean RMSE: {wf_df['RMSE_kb'].mean():.2f} kb")
print(f"Mean R²  : {wf_df['R2_Score'].mean():.4f}")
print(f"Mean Directional Accuracy: {wf_df['Directional_Acc_%'].mean():.1f}%")
print(f"\nResults saved directly to: {OUTPUT_DIR}")


=== Walk-Forward Validation Overall Averages ===
Mean MAE : 3619.01 kb
Mean RMSE: 4582.43 kb
Mean R²  : 0.0884
Mean Directional Accuracy: 62.5%

Results saved directly to: /content/drive/MyDrive/Tech Fem team - SIC capstone project/Week 2/4. Model Training and Optimization/4.5 Walk forward validation
